In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp '/content/drive/dataset_LSA-T'
!7z x "clips.7z" -o"clips"

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from transformers import CLIPModel, AutoProcessor, VideoMAEModel, VideoMAEImageProcessor
from PIL import Image
import cv2
from tqdm import tqdm
import shutil
from google.colab import drive 


def load_motion_model(device):
    print("Cargando VideoMAE...")
    processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-large")
    model = VideoMAEModel.from_pretrained("MCG-NJU/videomae-large").to(device)
    model.eval()  
    return model, processor

def load_clip_model(device):
    print("Cargando CLIP...")
    processor = AutoProcessor.from_pretrained("openai/clip-vit-large-patch14")
    model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14", device_map="auto").to(device)
    model.eval()  
    return model, processor

def handle_short_videos(num_frames, total_frames):
    if total_frames < num_frames:
        frame_indices = list(range(total_frames))
    else:
        frame_indices = set(np.linspace(0, total_frames - 1, num_frames, dtype=int))
    return frame_indices

def append_until_full(num_frames, frames):
    while len(frames) < num_frames:
        frames.append(Image.new('RGB', (224, 224), (0, 0, 0)))

def handle_frame_sampling(num_frames, cap, total_frames, frame_indices, frames):
    current_frame = 0
    
    while cap.isOpened() and len(frames) < num_frames:
        ret, frame = cap.read()
        if not ret:
            break
            
        if total_frames < num_frames or current_frame in frame_indices:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame))
            
        current_frame += 1
    cap.release()

def load_video_evenly(video_path, num_frames=16):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_indices = handle_short_videos(num_frames, total_frames)
    frames = []
    handle_frame_sampling(num_frames, cap, total_frames, frame_indices, frames)
    append_until_full(num_frames, frames)
    return frames

def load_cuda():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Usando: {device}")
    return device
  
def extract_all_features(device,spatial_dir,motion_dir):
    spatial_model, spatial_processor = load_clip_model(device)
    motion_model, motion_processor = load_motion_model(device)
    
    meta = pd.read_csv('meta.csv')

    os.makedirs(spatial_dir, exist_ok=True)
    os.makedirs(motion_dir, exist_ok=True)

    for idx, row in tqdm(meta.iterrows(), total=len(meta)):
        video_id = row['id']
        video_path = f"clips/{video_id}.mp4"
        
        if not os.path.exists(video_path):
            continue

        spatial_output = os.path.join(spatial_dir, f"{video_id}.npy")
        motion_output = os.path.join(motion_dir, f"{video_id}.npy")

        if os.path.exists(spatial_output) and os.path.exists(motion_output):
            continue

        frames = load_video_evenly(video_path, num_frames=16)
        if not frames:
            continue

        if not os.path.exists(spatial_output):
            extract_spatial_features(device, spatial_model, spatial_processor, frames, spatial_output)
            
        if not os.path.exists(motion_output):
            extract_motion_features(device, motion_model, motion_processor, frames, motion_output)

    print("\nDone!")

def extract_spatial_features(device, model, processor, frames, output_path):
    inputs = processor(images=frames, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        features = model.get_image_features(**inputs)  
        feats = features.cpu().numpy()
        
    np.save(output_path, feats)

def extract_motion_features(device, model, processor, frames, output_path):
    inputs = processor(images=frames, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        feats = outputs.hidden_states[-1][:, 0].cpu().numpy()
        
    np.save(output_path, feats)



def save_to_drive(source_dir):
    folder_name = os.path.basename(source_dir)
    drive_path = f'/content/drive/MyDrive/dataset_LSA-T{folder_name}'
    print(f"Copiando {source_dir} a {drive_path}...")
    try:
        shutil.copytree(source_dir, drive_path, dirs_exist_ok=True)
        print(f"¡Guardado exitoso en Drive: {drive_path}!")
    except Exception as e:
        print(f"Error al guardar en Drive: {e}")

def mount_drive():
    if not os.path.exists('/content/drive/MyDrive'):
      print("Montando Google Drive...")
      drive.mount('/content/drive')

def save_all_features_to_drive(spatial_dir, motion_dir):
    mount_drive()
    save_to_drive(spatial_dir)
    save_to_drive(motion_dir)

def save_all_features_to_drive(spatial_dir,motion_dir):
    save_to_drive(spatial_dir)
    save_to_drive(motion_dir)
if __name__ == "__main__":
    spatial_dir = './spatial_features_lsat'
    motion_dir = './motion_features_lsat'
    device = load_cuda()
    extract_all_features(device)
    save_all_features_to_drive(spatial_dir,motion_dir)